# AI6130 Assignment 2 -- LLM Fine-Tuning with PEFT (Kaggle)

**Covers all required + bonus tasks:**
- Models: `openlm-research/open_llama_3b_v2`, `TinyLlama/TinyLlama_v1.1`
- Methods: LoRA, AdapterP
- Train data: `math_7k.json` (required) + `math_10k.json` (bonus)
- Eval sets: AQuA, AddSub (required) + MultiArith, SingleEq, gsm8k, SVAMP (bonus)
- Epochs: 1 (required) + 2, 3 (bonus)

**Recommended Kaggle accelerator:** T4 x2 or P100 (16 GB VRAM)


In [1]:
import os
# 1. Move to working directory
os.chdir('/kaggle/working')

# 2. Clone the repo if it doesn't exist
if not os.path.exists('AI6130_Assignment2'):
    !git clone https://github.com/duyngtr16061999/AI6130_Assignment2

os.chdir('AI6130_Assignment2')

# 3. Install specific versions required by the assignment
!pip install fire datasets
!pip install transformers==4.38.0 accelerate==0.28.0
!pip uninstall peft -y  # Crucial: uses the local 'peft' folder in the repo instead

# 4. Silence WandB
os.environ["WANDB_MODE"] = "disabled"
print("✅ Environment is ready.")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working'

In [ ]:
!sed -i 's/use_cache=False/use_cache=True/g' /kaggle/working/AI6130_Assignment2/evaluate.py
print("🚀 High-speed inference (KV Cache) enabled!")

🚀 High-speed inference (KV Cache) enabled!


In [ ]:
import os

# 1. Setup Environment
os.chdir('/kaggle/working/AI6130_Assignment2')
os.environ["WANDB_MODE"] = "disabled"

models = [
    "openlm-research/open_llama_3b_v2",
    "TinyLlama/TinyLlama_v1.1"
]
adapters = [
    ("lora", "LoRA"),
    ("adapter", "AdapterP")
]
datasets = ["AddSub", "AQuA"]

print("🎬 STARTING CLEAN RUN: STANDARD ASSIGNMENT")

for model in models:
    for adapter_lower, adapter_eval in adapters:
        model_name = model.split('/')[-1]
        out_dir = f"./trained_models/{model_name}-{adapter_lower}"

        print(f"\n{'='*60}")
        print(f"🎯 TRAINING: {model_name} | Adapter: {adapter_eval}")
        print(f"{'='*60}")

        # Train (Using Single GPU to avoid device mismatch)
        train_cmd = (
            f"CUDA_VISIBLE_DEVICES=0 python finetune.py --base_model '{model}' "
            f"--data_path './ft-training_set/math_7k.json' "
            f"--output_dir '{out_dir}' --batch_size 4 --micro_batch_size 4 "
            f"--num_epochs 1 --learning_rate 3e-4 --cutoff_len 256 "
            f"--val_set_size 120 --adapter_name {adapter_lower}"
        )
        os.system(train_cmd)

        # Evaluate (Now with KV Cache enabled via the sed command earlier)
        for dataset in datasets:
            print(f"\n---> 🧪 EVALUATING: {model_name} | {adapter_eval} | {dataset}")
            eval_cmd = (
                f"CUDA_VISIBLE_DEVICES=0 python evaluate.py --adapter {adapter_eval} "
                f"--dataset {dataset} --base_model '{model}' "
                f"--lora_weights '{out_dir}'"
            )
            os.system(eval_cmd)

print("\n✨ ALL STANDARD EXPERIMENTS FINISHED!")

🎬 STARTING CLEAN RUN: STANDARD ASSIGNMENT

🎯 TRAINING: open_llama_3b_v2 | Adapter: LoRA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Finetuning model with params:
base_model: openlm-research/open_llama_3b_v2
data_path: ./ft-training_set/math_7k.json
output_dir: ./trained_models/open_llama_3b_v2-lora
batch_size: 4
micro_batch_size: 4
num_epochs: 1
learning_rate: 0.0003
cutoff_len: 256
val_set_size: 120
use_gradient_checkpointing: False
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target_modules: None
bottleneck_size: 256
non_linearity: tanh
adapter_dropout: 0.0
use_parallel_adapter: False
use_adapterp: False
train_on_inputs: True
scaling: 1.0
adapter_name: lora
target_modules: None
group_by_length: False
wandb_project: 
wandb_run_name: 
wandb_watch: 
wandb_log_model: 
resume_from_checkpoint: None



Generating train split: 6851 examples [00:00, 37325.10 examples/s]


trainable params: 2662400 || all params: 3429136000 || trainable%: 0.07764054852300988


...


---> 🧪 EVALUATING: TinyLlama_v1.1 | AdapterP | AddSub


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Traceback (most recent call last):
  File "/kaggle/working/AI6130_Assignment2/peft/src/peft/utils/config.py", line 101, in from_pretrained
    config_file = hf_hub_download(pretrained_model_name_or_path, CONFIG_NAME)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 154, in validate_repo_id
    raise HFValidationError(
huggingface_hub.errors.HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './trained_models/TinyLlam


---> 🧪 EVALUATING: TinyLlama_v1.1 | AdapterP | AQuA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Traceback (most recent call last):
  File "/kaggle/working/AI6130_Assignment2/peft/src/peft/utils/config.py", line 101, in from_pretrained
    config_file = hf_hub_download(pretrained_model_name_or_path, CONFIG_NAME)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 154, in validate_repo_id
    raise HFValidationError(
huggingface_hub.errors.HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './trained_models/TinyLlam


✨ ALL STANDARD EXPERIMENTS FINISHED!


In [ ]:
import os
os.chdir('/kaggle/working/AI6130_Assignment2')

print("🔍 Inspecting finetune.py around the error line...")
!cat -n finetune.py | grep -C 15 "get_peft_model"

print("\n🔍 Inspecting finetune.py for adapterp arguments...")
!grep -i "adapterp" finetune.py

In [ ]:
import os

# Ensure we are in the correct directory and WandB is disabled
os.chdir('/kaggle/working/AI6130_Assignment2')
os.environ["WANDB_MODE"] = "disabled"

# Bonus Experiment Parameters
model = "TinyLlama/TinyLlama_v1.1"
model_name = "TinyLlama_v1.1"
adapter_lower, adapter_eval = "lora", "LoRA"
out_dir = f"./trained_models/{model_name}-bonus-10k-3ep"
bonus_datasets = ["AddSub", "AQuA", "MultiArith", "SingleEq", "gsm8k", "SVAMP"]

print("\n🌟 STARTING BONUS EXPERIMENT...")
print(f"{'='*60}")
print(f"🎯 TRAINING: {model_name} | 3 Epochs | 10k Dataset")
print(f"{'='*60}")

# 1. Finetune (3 Epochs + 10k Dataset) - Forced Single GPU
train_cmd = (
    f"CUDA_VISIBLE_DEVICES=0 python finetune.py --base_model '{model}' "
    f"--data_path './ft-training_set/math_10k.json' "
    f"--output_dir '{out_dir}' --batch_size 4 --micro_batch_size 4 "
    f"--num_epochs 3 --learning_rate 3e-4 --cutoff_len 256 "
    f"--val_set_size 120 --adapter_name {adapter_lower}"
)
os.system(train_cmd)

# 2. Evaluate on ALL datasets - Forced Single GPU
for dataset in bonus_datasets:
    print(f"\n---> 🧪 EVALUATING BONUS: {dataset}")
    eval_cmd = (
        f"CUDA_VISIBLE_DEVICES=0 python evaluate.py --adapter {adapter_eval} "
        f"--dataset {dataset} --base_model '{model}' "
        f"--lora_weights '{out_dir}'"
    )
    os.system(eval_cmd)

print("\n✅ BONUS EXPERIMENT COMPLETE!")


🌟 STARTING BONUS EXPERIMENT...
🎯 TRAINING: TinyLlama_v1.1 | 3 Epochs | 10k Dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Finetuning model with params:
base_model: TinyLlama/TinyLlama_v1.1
data_path: ./ft-training_set/math_10k.json
output_dir: ./trained_models/TinyLlama_v1.1-bonus-10k-3ep
batch_size: 4
micro_batch_size: 4
num_epochs: 3
learning_rate: 0.0003
cutoff_len: 256
val_set_size: 120
use_gradient_checkpointing: False
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target_modules: None
bottleneck_size: 256
non_linearity: tanh
adapter_dropout: 0.0
use_parallel_adapter: False
use_adapterp: False
train_on_inputs: True
scaling: 1.0
adapter_name: lora
target_modules: None
group_by_length: False
wandb_project: 
wandb_run_name: 
wandb_watch: 
wandb_log_model: 
resume_from_checkpoint: None

trainable params: 1126400 || all params: 1101174784 || trainable%: 0.10229075496156657


Generating train split: 9919 examples [00:00, 58786.47 examples/s]
Map:   0%|          | 0/9799 [00:00<?, ? examples/s]Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} n


---> 🧪 EVALUATING BONUS: AddSub


...


---> 🧪 EVALUATING BONUS: gsm8k


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Traceback (most recent call last):
  File "/kaggle/working/AI6130_Assignment2/peft/src/peft/utils/config.py", line 101, in from_pretrained
    config_file = hf_hub_download(pretrained_model_name_or_path, CONFIG_NAME)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 154, in validate_repo_id
    raise HFValidationError(
huggingface_hub.errors.HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './trained_models/TinyLlam


---> 🧪 EVALUATING BONUS: SVAMP


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Traceback (most recent call last):
  File "/kaggle/working/AI6130_Assignment2/peft/src/peft/utils/config.py", line 101, in from_pretrained
    config_file = hf_hub_download(pretrained_model_name_or_path, CONFIG_NAME)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 154, in validate_repo_id
    raise HFValidationError(
huggingface_hub.errors.HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './trained_models/TinyLlam


✅ BONUS EXPERIMENT COMPLETE!
